<img align="right" src="images/tf.png" width="128"/>
<img align="right" src="images/logo.png" width="128"/>

# Tutorial

This notebook gets you started with using
[Text-Fabric](https://annotation.github.io/text-fabric/) for coding in Qumran-Digital.

Familiarity with the underlying
[data model](https://annotation.github.io/text-fabric/tf/about/datamodel.html)
is recommended.

## Installing Text-Fabric

See [here](https://annotation.github.io/text-fabric/tf/about/install.html)

## Tip
If you start computing with this tutorial, first copy its parent directory to somewhere else,
outside your repository.
If you pull changes from the repository later, your work will not be overwritten.
Where you put your tutorial directory is up to you.
It will work from any directory.

## Data

This tutorial works with a clone of the repository in `~/github/rdtaylorjr/qumran`.
Once the data is published on GitHub, Text-Fabric will fetch the data set for you
with `use("rdtaylorjr/qumran")`, and store it in the `text-fabric-data` in your home directory.

# Features
The data of the corpus is organized in features.
They are *columns* of data.
Think of the corpus as a gigantic spreadsheet, where row 1 corresponds to the
first word, row 2 to the second word, and so on, for all ~ 810 K words,
followed by ~ 250 K nodes of other types.

The information which text each word has, constitutes a column in that spreadsheet.
The Qumran-Digital corpus contains about 35 columns.

Instead of putting that information in one big table, the data is organized in separate columns.
We call those columns **features**.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import collections

# Incantation

The simplest way to get going is by this *incantation*:

In [ ]:
from tf.app import use

In [ ]:
A = use("rdtaylorjr/qumran:clone", checkout="clone", hoist=globals())

You can see which features have been loaded, and if you click on a feature name, you find its documentation.
If you hover over a name, you see where the feature is located on your system.

## API

The result of the incantation is that we have a bunch of special variables at our disposal
that give us access to the text and data of the corpus.

At this point it is helpful to throw a quick glance at the text-fabric API documentation
(see the links under **API Members** above).

The most essential thing for now is that we can use `F` to access the data in the features
we've loaded.
But there is more, such as `N`, which helps us to walk over the text, as we see in a minute.

The **API members** above show you exactly which new names have been inserted in your namespace.
If you click on these names, you go to the API documentation for them.

# Counting

In order to get acquainted with the data, we start with the simple task of counting.

## Count all nodes
We use the
[`N.walk()` generator](https://annotation.github.io/text-fabric/tf/core/nodes.html#tf.core.nodes.Nodes.walk)
to walk through the nodes.

In [ ]:
A.indent(reset=True)
A.info("Counting nodes ...")

i = 0
for n in N.walk():
    i += 1

A.info("{} nodes".format(i))

## What are those nodes?
Every node has a type, like word, part, line, column.
But what exactly are they?

Text-Fabric has two special features, `otype` and `oslots`, that must occur in every Text-Fabric data set.
`otype` tells you for each node its type, and you can ask for the number of `slot`s in the text.

Here we go!

In [ ]:
F.otype.slotType

In [ ]:
F.otype.maxSlot

In [ ]:
F.otype.maxNode

In [ ]:
F.otype.all

In [ ]:
C.levels.data

This is interesting: above you see all the textual objects, with the average size of their objects,
the node where they start, and the node where they end.

## Count individual object types
This is an intuitive way to count the number of nodes in each type.
Note in passing, how we use the `indent` in conjunction with `info` to produce neat timed
and indented progress messages.

In [ ]:
A.indent(reset=True)
A.info("counting objects ...")

for otype in F.otype.all:
    i = 0

    A.indent(level=1, reset=True)

    for n in F.otype.s(otype):
        i += 1

    A.info("{:>7} {}s".format(i, otype))

A.indent(level=0)
A.info("Done")

# Texts

Every transcription of Qumran-Digital is a *text*.
Its `kind` is its class in the index of Qumran-Digital: the non-biblical scrolls, the biblical scrolls,
and the books of the Masoretic Text and the Samaritan Pentateuch.
Here are the classes with the number of texts and words in each.

In [ ]:
texts = collections.Counter()
words = collections.Counter()

for t in F.otype.s("text"):
    texts[F.kind.v(t)] += 1
    words[F.kind.v(t)] += len(L.d(t, otype="word"))

for kind in texts:
    print(f"{kind:<13} {texts[kind]:>5} texts {words[kind]:>7} words")

# Sections

The sections are *text*, *column*, and *line*.
Columns are the columns or fragments of a manuscript, or the chapters of a book,
and lines are its lines or verses.

In [ ]:
c = T.nodeFromSection(("1QS", "5"))
T.sectionFromNode(c)

In [ ]:
lines = L.d(c, otype="line")
len(lines)

In [ ]:
A.plain(lines[0])

# Morphology

A word with a lexical analysis has a *part* for every part of the analysis,
such as the preposition and the noun of לאנשי.
The morphology is in the features of the BHSA.

In [ ]:
w = L.d(lines[0], otype="word")[2]
for p in sorted(L.u(w, otype="part"), key=lambda p: (F.priority.v(p), F.part.v(p))):
    print(F.priority.v(p), F.part.v(p), F.lex.v(p), F.sp.v(p), F.gn.v(p), F.nu.v(p), F.st.v(p))

Where a source label has no BHSA value of the same meaning, its abbreviation is kept,
such as `det.` for a noun with the article.

In [ ]:
F.st.freqList()

In [ ]:
F.vs.freqList()

# Lexemes

Every lemma is a *lex* node over the words that have it, with its German short definition.

In [ ]:
for x in L.u(w, otype="lex"):
    print(F.lex.v(x), Fs("gloss@de").v(x), len(L.d(x, otype="word")))

# Parallels

The edge `parallel` links a line to its parallel lines, and a word to the parallel words aligned with it.
Here are the parallels of Genesis 1:1.

In [ ]:
v = T.nodeFromSection(("Gen", "1", "1"))
for p in E.parallel.f(v):
    print(T.sectionFromNode(p), T.text(p))

And here are the words of 1QS 5:1 beside the words of 4Q256 and 4Q258 aligned with them.

In [ ]:
def aligned(w):
    return {F.text.v(L.u(p, otype="text")[0]): F.text.v(p) for p in E.parallel.f(w)}


for w in L.d(lines[0], otype="word"):
    found = aligned(w)
    print(f"{F.text.v(w):<12} {found.get('4Q256', ''):<12} {found.get('4Q258', '')}")

# Syntax

The edge `head` links a word to the word it depends on, with the syntactic category of the dependent.

In [ ]:
for w in L.d(lines[0], otype="word"):
    for head, category in E.head.f(w):
        print(f"{F.text.v(w):<12} {category:<16} {F.text.v(head)}")

# Search

Text-Fabric has a search engine.
Here we look for verbs in the hifil in 1QS.
A part has the same slot as its word, so it goes directly under its text.

In [ ]:
query = """
text text=1QS
  part sp=verb vs=hif
"""
results = A.search(query)

In [ ]:
A.table(results, end=5)

# All steps

* **start** your first step in mastering Qumran-Digital computationally
* **[search](search.ipynb)** turbo charge your hand-coding with search templates